# Benchmarks - Python

The one Python example from [docs/benchmarks.md](https://platob.github.io/yggdryl/benchmarks/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

### The pipeline those numbers measure

In [ ]:
import gzip
import pathlib
import tempfile

import pyarrow as pa
import pyarrow.compute as pc

from yggdryl import IOBase

pattern = (
    r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\S*"
    r" \[(?<level>[^\]]+)\] \[(?<logger>[^\]]+)\]"
    r" \[(?<thread_id>\d+)\] took=(?<latency_us>\d+)"
)

# Two rotated leaves; the second record of the first spans a stack trace.
leaves = [
    "2024-02-01 10:00:00.000000 [ii] [engine] [3] took=120 fill 100 SYMB-0001\n"
    "2024-02-01 10:00:01.000000 [ee] [engine] [4] took=980 fill 101 SYMB-0002\n"
    "    at engine::match(order.rs:118)\n"
    "    at engine::step(order.rs:64)\n"
    "2024-02-01 10:00:02.000000 [ww] [router] [5] took=240 fill 102 SYMB-0003\n",
    "2024-02-01 10:00:03.000000 [ee] [ledger] [6] took=770 fill 103 SYMB-0004\n"
    "2024-02-01 10:00:04.000000 [ii] [feed] [7] took=100 fill 104 SYMB-0005\n",
]
root = pathlib.Path(tempfile.mkdtemp())
for index, text in enumerate(leaves):
    (root / f"app-{index}.log.gz").write_bytes(gzip.compress(text.encode()))

rows = errors = traced = 0
latency = 0
# The reader is lazy: one batch crosses at a time, over the C Stream.
for batch in IOBase(root).read_arrow_lines(pattern):
    rows += batch.num_rows
    traced += pc.sum(pc.cast(pc.greater(batch.column("lines"), 1), pa.int64())).as_py()
    kept = batch.filter(pc.equal(batch.column("level"), "ee"))
    errors += kept.num_rows
    if kept.num_rows:
        # `latency_us` is int64 already, so this is a sum, not a parse.
        latency += pc.sum(kept.column("latency_us")).as_py()

assert (rows, errors, traced) == (5, 2, 1)
assert latency == 980 + 770
assert IOBase(root).read_arrow_lines(pattern).schema.field("latency_us").type == pa.int64()